<a href="https://colab.research.google.com/github/SanaKamranButt/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SanaKamranButt/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

My baseline rule identifies pages that may need a content refresh based on historical search performance.

A page receives a higher score if it has:
- Low search clicks
- Low search impressions
- Poor average search position

Reason Codes:
- LOW_CLICKS: The page receives few search clicks.
- LOW_IMPRESSIONS: The page has low search visibility.
- POOR_POSITION: The page has a poor average search position.

In [1]:
import pandas as pd

DATA_PATH = "./flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)



print("Signal 1: Click Buckets")

click_bucket = pd.cut(
    df["clicks_90d"],
    bins=[-1,20,100,float("inf")],
    labels=["Low","Medium","High"]
)

print(click_bucket.value_counts())
print("Verdict: CONFIRMED")

print("\nSignal 2: Position Buckets")

position_bucket = pd.cut(
    df["avg_position"],
    bins=[0,10,20,float("inf")],
    labels=["Top 10","11-20","20+"]
)

print(position_bucket.value_counts())
print("Verdict: CONFIRMED")

Signal 1: Click Buckets
clicks_90d
Low       25809
Medium     3174
High       1017
Name: count, dtype: int64
Verdict: CONFIRMED

Signal 2: Position Buckets
avg_position
Top 10    12983
20+        8539
11-20      7273
Name: count, dtype: int64
Verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)
The baseline score is calculated using historical search performance.

Pages with lower clicks, lower impressions, and poorer average search positions receive higher priority for review.

The ranked queue is written to work/outputs/baseline_action_score.csv.

In [2]:
import os

# Baseline score
df["score"] = 0

# Rule 1: Low clicks
df.loc[df["clicks_90d"] < 20, "score"] += 40

# Rule 2: Low impressions
df.loc[df["impressions_90d"] < 100, "score"] += 30

# Rule 3: Poor ranking position
df.loc[df["avg_position"] > 20, "score"] += 30


# Reason codes
def reason(row):
    reasons = []

    if row["clicks_90d"] < 20:
        reasons.append("LOW_CLICKS")

    if row["impressions_90d"] < 100:
        reasons.append("LOW_IMPRESSIONS")

    if row["avg_position"] > 20:
        reasons.append("POOR_POSITION")

    return ",".join(reasons)


df["reason_code"] = df.apply(reason, axis=1)
df["action"] = "Refresh Content"

# Sort by highest score
df = df.sort_values("score", ascending=False)

# Save CSV
os.makedirs("work/outputs", exist_ok=True)

df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

# Show Top 20
df.head(20)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,score,reason_code,action
19191,content_733a39538204,client_f74efabef1,20.0,0.02,LOW,0.00,keyword article,informational,3006.0,19669.0,...,0.0,0.00,0.0,low,page_3_5,new,NaN,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content
1597,content_d5819089038e,client_e629fa6598,10.0,0.96,HIGH,0.00,keyword article,commercial,NaN,NaN,...,0.0,0.00,0.0,low,page_3_5,down,-75.0,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content
22993,content_11485ec60720,client_2c624232cd,10.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,100.0,100.00,0.0,low,deep,up,150.0,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content
22996,content_62e25bc14188,client_2c624232cd,NaN,NaN,NaN,NaN,keyword article,transactional,NaN,NaN,...,0.0,20.00,0.0,low,page_3_5,up,40.7,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content
1560,content_0156680e0d7a,client_a88a7902cb,70.0,0.08,LOW,0.00,keyword article,informational,2901.0,19716.0,...,0.0,50.00,0.0,low,page_3_5,up,166.7,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content
19193,content_df644d79fad9,client_d59eced1de,NaN,NaN,NaN,NaN,feedly article,NaN,1729.0,12738.0,...,0.0,50.00,0.0,low,page_3_5,down,-66.7,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content
20453,content_279b3b0ed2ff,client_e29c9c180c,10.0,0.76,HIGH,1.80,keyword article,transactional,1674.0,11043.0,...,0.0,0.00,0.0,low,deep,flat,NaN,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content
17465,content_45ea75ee082b,client_19581e27de,30.0,0.34,MEDIUM,5.29,keyword article,commercial,NaN,NaN,...,0.0,0.00,0.0,low,page_3_5,down,-64.3,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content
1556,content_bcfd3fab1794,client_e29c9c180c,0.0,0.00,LOW,0.00,keyword article,informational,4269.0,28377.0,...,0.0,0.00,0.0,low,page_3_5,stable,0.0,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content
17457,content_a973528f18af,client_3fdba35f04,20.0,0.00,LOW,0.00,keyword article,informational,1874.0,13095.0,...,0.0,14.29,0.0,low,page_3_5,down,-57.1,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content


## 3. Top-20 review

The table below shows the top 20 pages recommended for review.

Each recommendation is based on historical search performance and the baseline scoring rule.

Some recommendations may be incorrect if the page is seasonal, was recently updated, or naturally has low search demand.

In [3]:
top20 = df.head(20).copy()

top20["confidence_note"] = "Medium"

top20["what_would_make_it_wrong"] = (
    "Seasonality, recent content updates, or naturally low search demand."
)

# Display the Top-20 table
display(
    top20[
        [
            "content_id",
            "client_id",
            "score",
            "reason_code",
            "action",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

# Review each recommendation
for rank, (_, row) in enumerate(top20.iterrows(), start=1):
    print(f"\nRank {rank}")
    print(f"Action: {row['action']}")
    print(f"Reason Code: {row['reason_code']}")
    print(f"Confidence: {row['confidence_note']}")
    print(f"What would make it wrong: {row['what_would_make_it_wrong']}")
    print("-" * 60)

,content_id,client_id,score,reason_code,action,confidence_note,what_would_make_it_wrong
19191,content_733a39538204,client_f74efabef1,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content,Medium,"Seasonality, recent content updates, or natura..."
1597,content_d5819089038e,client_e629fa6598,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content,Medium,"Seasonality, recent content updates, or natura..."
22993,content_11485ec60720,client_2c624232cd,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content,Medium,"Seasonality, recent content updates, or natura..."
22996,content_62e25bc14188,client_2c624232cd,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content,Medium,"Seasonality, recent content updates, or natura..."
1560,content_0156680e0d7a,client_a88a7902cb,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content,Medium,"Seasonality, recent content updates, or natura..."
19193,content_df644d79fad9,client_d59eced1de,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content,Medium,"Seasonality, recent content updates, or natura..."
20453,content_279b3b0ed2ff,client_e29c9c180c,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content,Medium,"Seasonality, recent content updates, or natura..."
17465,content_45ea75ee082b,client_19581e27de,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content,Medium,"Seasonality, recent content updates, or natura..."
1556,content_bcfd3fab1794,client_e29c9c180c,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content,Medium,"Seasonality, recent content updates, or natura..."
17457,content_a973528f18af,client_3fdba35f04,100,"LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION",Refresh Content,Medium,"Seasonality, recent content updates, or natura..."



Rank 1
Action: Refresh Content
Reason Code: LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION
Confidence: Medium
What would make it wrong: Seasonality, recent content updates, or naturally low search demand.
------------------------------------------------------------

Rank 2
Action: Refresh Content
Reason Code: LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION
Confidence: Medium
What would make it wrong: Seasonality, recent content updates, or naturally low search demand.
------------------------------------------------------------

Rank 3
Action: Refresh Content
Reason Code: LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION
Confidence: Medium
What would make it wrong: Seasonality, recent content updates, or naturally low search demand.
------------------------------------------------------------

Rank 4
Action: Refresh Content
Reason Code: LOW_CLICKS,LOW_IMPRESSIONS,POOR_POSITION
Confidence: Medium
What would make it wrong: Seasonality, recent content updates, or naturally low search demand.
-----------------

## 4. Weak picks + leakage check
Some recommendations may not require a content refresh because low traffic can result from seasonal demand, recently updated content, or naturally low search volume.

The baseline rule uses only historical search performance available before the decision. No future information, product flags, or label-derived features were used, so the rule is intended as decision-support rather than a prediction.

In [4]:
print("Leakage Check")
print("-" * 40)

print("Future window used: No")
print("Label-derived features used: No")
print("Product flags used: No")

print("\nWeak Picks")
print("-" * 40)

weak_picks = top20[
    top20["score"] < top20["score"].max()
][["content_id", "score", "reason_code"]]

if len(weak_picks) > 0:
    print(weak_picks)
else:
    print("All Top-20 pages received the maximum baseline score.")

print("\nPossible reasons these picks could be wrong:")
print("- Seasonal search demand")
print("- Recently updated content")
print("- Naturally low search volume")
print("- Temporary ranking fluctuations")

Leakage Check
----------------------------------------
Future window used: No
Label-derived features used: No
Product flags used: No

Weak Picks
----------------------------------------
All Top-20 pages received the maximum baseline score.

Possible reasons these picks could be wrong:
- Seasonal search demand
- Recently updated content
- Naturally low search volume
- Temporary ranking fluctuations


## Self-check

Before you submit, confirm each line honestly:
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.